# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RehanTajik-730/ML_INTERNSHIP/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton establishes the transparent, rule-based baseline for **Lane 4 (CTR / Engagement Opportunity Scoring)**. Before training a machine learning model, we must verify that the underlying search signals are real, encode a deterministic rule that produces actionable recommendations, and critically audit the top of the resulting queue with a skeptic's eye.

> **Lane Confirmation**: Lane 4 — CTR / Engagement Opportunity Scoring (Confirmed).

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Plain-Language Rule Definition
A content page is flagged as an urgent CTR optimization candidate if it satisfies three transparent, observable search criteria:
1. **High Visibility**: It generates substantial search demand ($\ge 500$ impressions in the trailing 90 days), filtering out small-sample denominator noise.
2. **Striking Distance Position**: It ranks on Google Page 1 or top of Page 2 ($0 < \text{avg\_position} \le 20$), where searchers actively browse and where snippet improvements directly translate into user clicks.
3. **Severe CTR Underperformance**: Its captured click-through rate is below the baseline benchmark ($\text{CTR} < 0.50\%$).

When all three conditions hold, the rule flags the page, assigns a single reason code, recommends an action, and calculates a volume-weighted opportunity score representing estimated uncaptured clicks (click deficit):
$$\text{Baseline Score} = \text{rule\_flag} \times \left(\frac{0.50 - \text{CTR}}{100}\right) \times \text{impressions\_90d}$$
Pages not meeting all three criteria receive a score of `0.0`.

### Reason Codes and Action Labels
- **ONE Reason Code**: `low_ctr_visible_page` (assigned to every page meeting the rule; unflagged pages receive `none`).
- **ONE Action Label**: `review_metadata_and_serp` (actionable recommendation: audit meta title, snippet description, search intent alignment, and SERP layout; unflagged pages receive `monitor`).

---

### Signal Check 1 (Flag-Linked): CTR vs. Position Tier
- **Underlying FlyRank Flag**: `low_ctr_visible_page` (CTR-fix logic).
- **Core Hypothesis**: Searchers click top results at significantly higher rates than lower results. As average position worsens, observed CTR must decay monotonically. Comparing raw CTR without conditioning on position tier is misleading; a 0.5% CTR is an underperformance on Page 1 ($1 \le \text{pos} \le 10$) but expected on Page 3+.
- **Verdict**: **CONFIRMED** — Both mean and volume-weighted CTR decay strictly and monotonically across position tiers from $2.71\%$ in top-3 down to $0.15\%$ in deep rankings ($>50$).

### Signal Check 2 (Flag-Linked): Impression Volume vs. Click Capture & Denominator Noise
- **Underlying FlyRank Flag**: Quick-win minimum volume gating ($\text{impressions} \ge 500$).
- **Core Hypothesis**: At low impression counts ($< 500$), observed CTR is corrupted by severe denominator noise (e.g. 1 click on 2 impressions yields 50% CTR; 89% of pages have 0 clicks). Setting a minimum volume threshold of $\ge 500$ impressions isolates pages with reliable click potential where metadata fixes yield measurable traffic lift.
- **Verdict**: **CONFIRMED** — Pages with $<100$ impressions suffer an $89.2\%$ zero-click rate, whereas pages with $\ge 500$ impressions establish solid demand ($<30\%$ zero-click rate) and substantial click capture (mean 2.7 to 103 clicks).

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve repository root
cwd = Path(os.getcwd())
if (cwd / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    repo_root = cwd
elif (cwd.parent.parent / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    repo_root = cwd.parent.parent
elif (cwd.parent / "data" / "raw" / "content_refresh_anonymized.csv").exists():
    repo_root = cwd.parent
else:
    repo_root = Path(".")

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)
print(f"Loaded dataset: {len(df):,} total rows from {data_path.name}\n")

# -------------------------------------------------------------------------
# SIGNAL CHECK 1: CTR vs Position Tier (Excluding avg_position == 0 no-data rows)
# -------------------------------------------------------------------------
valid_pos = df[df["avg_position"] > 0].copy()
valid_pos["pos_tier_clean"] = pd.cut(
    valid_pos["avg_position"],
    bins=[0, 3, 10, 20, 50, 500],
    labels=["top_3 (1-3)", "page_1 (4-10)", "striking (11-20)", "page_3_5 (21-50)", "deep (>50)"]
)

sig1_table = valid_pos.groupby("pos_tier_clean", observed=False).agg(
    n=("content_id", "count"),
    mean_position=("avg_position", "mean"),
    mean_ctr_pct=("ctr", "mean"),
    median_ctr_pct=("ctr", "median"),
    weighted_ctr_pct=("clicks_90d", lambda x: (x.sum() / valid_pos.loc[x.index, "impressions_90d"].sum()) * 100)
).reset_index()

print("=" * 80)
print("SIGNAL 1 BUCKET TABLE: CTR Decay Across Position Tiers")
print("=" * 80)
print(sig1_table.to_string(index=False))
print("-" * 80)
print("Signal 1 Verdict: CONFIRMED")
print("Rationale: Mean CTR declines monotonically from 2.71% in top-3 to 0.65% on page 1,")
print("0.32% in striking distance, and 0.15% in deep positions. Expected CTR is strictly rank-dependent.\n")

# -------------------------------------------------------------------------
# SIGNAL CHECK 2: Impression Volume vs Click Reliability & Denominator Noise
# -------------------------------------------------------------------------
df["vol_tier"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 2500, 10000, 1000000],
    labels=["<100 (noise)", "100-499 (low)", "500-2499 (moderate)", "2500-9999 (high)", "10000+ (head)"]
)

sig2_table = df.groupby("vol_tier", observed=False).agg(
    n=("content_id", "count"),
    mean_impressions=("impressions_90d", "mean"),
    mean_clicks=("clicks_90d", "mean"),
    zero_click_pct=("clicks_90d", lambda x: (x == 0).mean() * 100),
    mean_ctr_pct=("ctr", "mean"),
    weighted_ctr_pct=("clicks_90d", lambda x: (x.sum() / df.loc[x.index, "impressions_90d"].sum()) * 100)
).reset_index()

print("=" * 80)
print("SIGNAL 2 BUCKET TABLE: Impression Volume vs Click Reliability")
print("=" * 80)
print(sig2_table.to_string(index=False))
print("-" * 80)
print("Signal 2 Verdict: CONFIRMED")
print("Rationale: Pages with <100 impressions suffer from 89.2% zero-click rates and noisy rates.")
print("At impressions >= 500, zero-click rates plunge below 30% and clicks reach meaningful scale (mean 2.7 to 103 clicks).\n")

Loaded dataset: 30,000 total rows from content_refresh_anonymized.csv

SIGNAL 1 BUCKET TABLE: CTR Decay Across Position Tiers
  pos_tier_clean     n  mean_position  mean_ctr_pct  median_ctr_pct  weighted_ctr_pct
     top_3 (1-3)  1141       2.121735      2.714303            0.00          0.489931
   page_1 (4-10) 11842       6.596149      0.651045            0.16          0.349063
striking (11-20)  7273      14.308360      0.323443            0.10          0.348607
page_3_5 (21-50)  7225      30.719765      0.222345            0.03          0.153781
      deep (>50)  1314      63.716819      0.150784            0.00          0.041369
--------------------------------------------------------------------------------
Signal 1 Verdict: CONFIRMED
Rationale: Mean CTR declines monotonically from 2.71% in top-3 to 0.65% on page 1,
0.32% in striking distance, and 0.15% in deep positions. Expected CTR is strictly rank-dependent.

SIGNAL 2 BUCKET TABLE: Impression Volume vs Click Reliability
     

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Rule Implementation Mechanics
We encode the hand-written rule cleanly using transparent conditions with no fitted weights:
- `visible = (impressions_90d >= 500)`
- `in_striking = (avg_position > 0) & (avg_position <= 20)`
- `low_ctr = (ctr < 0.50)`
- `rule_flag = visible * in_striking * low_ctr`

### The Scoring Formula
For every page meeting the rule, the baseline opportunity score quantifies the estimated lost clicks resulting from underperforming below the 0.50% CTR floor:
$$\text{baseline\_score} = \text{rule\_flag} \times \text{impressions\_90d} \times \frac{0.50 - \text{ctr}}{100}$$
Unflagged pages receive a score of `0.0`.

### Output Artifacts
The notebook generates:
1. `work/outputs/baseline_action_score.csv`: The full prioritized queue of all 30,000 pages sorted by `baseline_score` descending (with secondary sort by `impressions_90d`), carrying rank, reason code (`low_ctr_visible_page`), and action label (`review_metadata_and_serp`).
2. `work/outputs/baseline_metrics.json`: Run receipts recording dataset dimensions, candidate counts, score distributions, and signal summaries.

In [2]:
import json

out_dir = repo_root / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Evaluate hand-written rule conditions
visible = (df["impressions_90d"] >= 500).astype(int)
in_striking = ((df["avg_position"] > 0) & (df["avg_position"] <= 20)).astype(int)
low_ctr = (df["ctr"] < 0.50).astype(int)
rule_flag = visible * in_striking * low_ctr

# 2. Compute transparent baseline score (estimated click deficit)
df["baseline_score"] = (rule_flag * (df["impressions_90d"] * (0.50 - df["ctr"]) / 100)).round(4)

# 3. Assign single reason code and action label
df["reason_code"] = np.where(rule_flag == 1, "low_ctr_visible_page", "none")
df["action_label"] = np.where(rule_flag == 1, "review_metadata_and_serp", "monitor")

# 4. Rank entire inventory
queue = df.sort_values(by=["baseline_score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["baseline_rank"] = queue.index + 1

# 5. Export ranked queue to CSV
output_columns = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "avg_position",
    "ctr",
    "clicks_90d",
    "main_intent",
    "content_type"
]
csv_path = out_dir / "baseline_action_score.csv"
queue[output_columns].to_csv(csv_path, index=False)
print(f"Exported ranked queue: {csv_path} ({csv_path.stat().st_size:,} bytes)")

# 6. Export metrics receipts to JSON
flagged_count = int(rule_flag.sum())
flagged_df = queue[queue["baseline_score"] > 0]

metrics = {
    "total_pages_scored": int(len(queue)),
    "flagged_candidates_count": flagged_count,
    "flagged_pct": round(flagged_count / len(queue) * 100, 2),
    "score_max": float(queue["baseline_score"].max()),
    "score_top50_mean": float(queue.head(50)["baseline_score"].mean()),
    "score_flagged_median": float(flagged_df["baseline_score"].median()),
    "rule_definition": {
        "impressions_floor": 500,
        "position_range": [0.01, 20.0],
        "ctr_threshold_pct": 0.50,
        "reason_code": "low_ctr_visible_page",
        "action_label": "review_metadata_and_serp"
    },
    "signals_checked": {
        "signal_1": {"name": "CTR decay across position tiers", "verdict": "CONFIRMED"},
        "signal_2": {"name": "Impression volume vs click reliability", "verdict": "CONFIRMED"}
    }
}
json_path = out_dir / "baseline_metrics.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print(f"Exported run receipt: {json_path}")

print(f"\nQueue Summary:")
print(f"  Total Inventory Scored: {len(queue):,} pages")
print(f"  Flagged Opportunities:  {flagged_count:,} pages ({flagged_count / len(queue) * 100:.1f}%)")
print(f"  Top Score (Lost Clicks): {queue['baseline_score'].max():,.2f}")
print(f"  Median Score (Flagged):  {flagged_df['baseline_score'].median():,.2f}")

Exported ranked queue: C:\Users\Rehan Ali\Documents\antigravity\gallant-galileo\work\outputs\baseline_action_score.csv (3,554,513 bytes)
Exported run receipt: C:\Users\Rehan Ali\Documents\antigravity\gallant-galileo\work\outputs\baseline_metrics.json

Queue Summary:
  Total Inventory Scored: 30,000 pages
  Flagged Opportunities:  9,759 pages (32.5%)
  Top Score (Lost Clicks): 1,863.77
  Median Score (Flagged):  8.67


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing the top candidates with a skeptic's eye is where simplistic rule logic is stress-tested. Below is the row-by-row audit for the top 10 review (and expanded to top 20), stating the action, why each page is ranked here, and the exact search/SERP conditions that would make the recommendation wrong.

### Top-10 Skeptic Audit (Action, Why It's There, What Would Make It Wrong)
1. **Rank 1 (`content_5fe46e04994d`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
   - *Why it's there*: Generates massive visibility ($517,715$ impressions) ranking at position $4.2$ (upper Page 1), but achieves only $0.14\%$ CTR ($741$ clicks) vs the $0.50\%$ benchmark, representing an estimated deficit of $1,864$ clicks.
   - *What would make it wrong*: The query is a broad informational search with an instant Google AI Overview or Featured Snippet that satisfies user intent directly on the SERP without requiring an organic click.
2. **Rank 2 (`content_8c19996aa890`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
   - *Why it's there*: Occupies prime search real estate at position $2.5$ with $509,252$ impressions, yet captures an abnormally weak $0.15\%$ CTR ($785$ clicks), calculating an opportunity gap of $1,782$ uncaptured clicks.
   - *What would make it wrong*: The query contains dominant navigational brand intent for a competitor; searchers deliberately bypass non-brand results regardless of snippet quality.
3. **Rank 3 (`content_36ff89c8214e`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
   - *Why it's there*: Holds a strong Page 1 ranking ($7.3$) with $295,097$ impressions, but yields a microscopic $0.05\%$ CTR ($154$ clicks), giving an estimated deficit of $1,328$ clicks.
   - *What would make it wrong*: Position 7.3 sits below a heavy stack of Google Ads, a local pack, and a People Also Ask accordion, visually displacing the organic result far down the page.
4. **Rank 4 (`content_aaef01a50def`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
   - *Why it's there*: Enormous search volume ($517,109$ impressions) at position $5.4$ with below-par $0.25\%$ CTR ($1,270$ clicks), pointing to a potential gain of $1,293$ clicks through improved snippet appeal.
   - *What would make it wrong*: Searchers seek interactive tools or downloadable files rather than long-form articles, rendering title tag optimization ineffective without format changes.
5. **Rank 5 (`content_8451fc6f034d`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
   - *Why it's there*: Position $2.3$ with $272,144$ impressions but only $0.03\%$ CTR ($75$ clicks), producing an estimated deficit of $1,279$ clicks.
   - *What would make it wrong*: Extreme navigational mismatch — searchers are seeking a login portal or utility page, so an informational article will never capture clicks at this rank.
6. **Rank 6 (`content_1a9e894be2e2`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
   - *Why it's there*: High transactional demand ($416,180$ impressions) at position $4.0$ with $0.23\%$ CTR ($944$ clicks), calculating an estimated $1,124$ lost clicks against benchmark.
   - *What would make it wrong*: The SERP is dominated by Google Shopping product cards that absorb commercial click intent before users ever view organic listings.
7. **Rank 7 (`content_cb112fce36be`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
   - *Why it's there*: Mid-Page-1 position ($5.6$) with $309,910$ impressions and $0.16\%$ CTR ($492$ clicks), generating a deficit of $1,054$ clicks.
   - *What would make it wrong*: Competing listings feature rich snippets (star ratings, review counts, pricing) that naturally command user attention despite a sound title.
8. **Rank 8 (`content_c84a0ab98e90`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
   - *Why it's there*: $223,271$ impressions at position $7.8$ capturing a negligible $0.03\%$ CTR ($70$ clicks), indicating an estimated $1,049$ click deficit.
   - *What would make it wrong*: Ambiguous search intent where Google serves mixed categories, leaving our specific sub-topic relevant to only a tiny fraction of search impressions.
9. **Rank 9 (`content_c8e9d6ab9013`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
   - *Why it's there*: Position $9.7$ with $208,678$ impressions and exactly $0$ clicks ($0.00\%$ CTR), producing a computed gap of $1,043$ clicks.
   - *What would make it wrong*: Sits right at the bottom edge of Page 1 ($9.7$), where impressions may be inflated by automated search scrapers/rank trackers rather than actual human readers scrolling to the bottom.
10. **Rank 10 (`content_db5989a78dd3`)** — Action: `review_metadata_and_serp` (Reason: `low_ctr_visible_page`)
    - *Why it's there*: Commercial query page with $345,111$ impressions at position $5.4$ and $0.21\%$ CTR ($733$ clicks), showing $1,001$ estimated lost clicks.
    - *What would make it wrong*: Searchers demand pricing transparently in the snippet, but client business constraints prevent disclosing prices publicly in search metadata.

### Ranks 11–20 Overview
- **Ranks 11–13** (`content_73c54f78c06a`, `content_a7427266c305`, `content_91652435f57a`): Strong Page 1 rankings (pos 4.7–7.8) with 159k–214k impressions and CTRs between 0.06% and 0.11%. High confidence review candidates, vulnerable primarily to rich competitor schema.
- **Rank 14 & 19** (`content_453722754fea`, `content_4a6607efcb46`): Extreme low-CTR outliers ($0.01\%$ CTR) at positions 7.6 and 2.2 with over 128k impressions and under 20 clicks. Highly suspected navigational intent or bot impressions.
- **Ranks 15–18, 20** (`content_e12868d1f396`, `content_2db251d1a841`, `content_97a86caf3a3d`, `content_c1fe78bc4e37`, `content_33b4dceecad1`): High-volume informational and transactional opportunities across positions 2.9 to 7.5 with calculated deficits of 617 to 644 clicks.

In [3]:
top20_display = queue.head(20)[[
    "baseline_rank",
    "content_id",
    "baseline_score",
    "impressions_90d",
    "avg_position",
    "ctr",
    "clicks_90d",
    "main_intent",
    "content_type",
    "reason_code",
    "action_label"
]]

print("=" * 115)
print("TOP 20 BASELINE REVIEW QUEUE (Ranked by Click Opportunity Deficit)")
print("=" * 115)
print(top20_display.to_string(index=False))
print("-" * 115)

TOP 20 BASELINE REVIEW QUEUE (Ranked by Click Opportunity Deficit)
 baseline_rank           content_id  baseline_score  impressions_90d  avg_position  ctr  clicks_90d   main_intent    content_type          reason_code             action_label
             1 content_5fe46e04994d       1863.7740           517715           4.2 0.14         741 informational keyword article low_ctr_visible_page review_metadata_and_serp
             2 content_8c19996aa890       1782.3820           509252           2.5 0.15         785 informational keyword article low_ctr_visible_page review_metadata_and_serp
             3 content_36ff89c8214e       1327.9365           295097           7.3 0.05         154 informational keyword article low_ctr_visible_page review_metadata_and_serp
             4 content_aaef01a50def       1292.7725           517109           5.4 0.25        1270 informational keyword article low_ctr_visible_page review_metadata_and_serp
             5 content_8451fc6f034d       1279.0768  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Skeptical Analysis: The Weak Picks
A mechanical rule that relies solely on volume, position, and CTR will inevitably promote false positives that look lucrative in math but fail in real search dynamics. Our review identified three distinct types of weak picks in the top tier:

1. **The Ghost Impression Anomaly (Rank 9: `content_c8e9d6ab9013`)**
   - *Observation*: $208,678$ impressions, average position $9.7$, and literally $0$ clicks ($0.00\%$ CTR).
   - *Why it's weak*: Position $9.7$ is at the bottom boundary of Page 1. In organic search, position 9.7 receives impressions whenever anyone searches, but almost no human eyes scan down to the 10th result before bouncing. Additionally, automated rank scrapers poll search engines, registering impressions on page 1 results without ever generating clicks. Spending editorial time rewriting metadata for a position 9.7 page with 0 clicks is a low-probability gamble compared to optimizing position 3–5 pages.
2. **The Navigational Identity Trap (Rank 5: `content_8451fc6f034d` and Rank 19: `content_4a6607efcb46`)**
   - *Observation*: High rankings ($2.3$ and $2.2$) with massive impressions ($272\text{k}$ and $128\text{k}$) but essentially flat click counts ($75$ and $17$ clicks; CTRs of $0.03\%$ and $0.01\%$).
   - *Why it's weak*: Top-3 organic positions average $>2\%$ CTR across the dataset. A CTR of $0.01\%$ at position $2.2$ strongly indicates that the search term is a navigational brand search (e.g. searching for a login portal, customer service phone number, or government site). Users already know where they want to go; no content meta-title rewrite will convince them to click an informational article instead.
3. **Zero-Click SERP Displacement (Rank 1: `content_5fe46e04994d`)**
   - *Observation*: $517,715$ impressions at position $4.2$ with only $0.14\%$ CTR.
   - *Why it's weak*: Informational head queries often trigger Google AI Overviews or Direct Answer boxes. If the SERP answers the question directly, impressions register but organic clicks evaporate across all listings.

---

### Strict Leakage Check
We strictly verified that no forbidden or leaking signals were introduced into the baseline scoring logic:

1. **No Product Flags or Decisions**: FlyRank's existing product outputs (`health_score`, `priority_score`, `action_type`) were excluded entirely from feature calculations.
2. **No Future-Window Peeking**: Only trailing 90-day observable historical metrics were used (`impressions_90d`, `avg_position`, `ctr`). No future or post-split windows (`impressions_last_30d`, `clicks_last_30d`) were accessed.
3. **No Target-Derived Inputs**: The model targets (`trend_direction`, `trend_pct`, `is_declining_label`) were never used as features or score components.

In [4]:
# -------------------------------------------------------------------------
# PROGRAMMATIC LEAKAGE AND INTEGRITY CHECK
# -------------------------------------------------------------------------
print("=" * 80)
print("LEAKAGE AND INTEGRITY AUDIT")
print("=" * 80)

# Check 1: Verify inputs used in scoring
allowed_inputs = {"impressions_90d", "avg_position", "ctr"}
prohibited_candidates = [
    "health_score",
    "priority_score",
    "action_type",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d"
]

detected_leaks = [
    col for col in prohibited_candidates
    if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
    and np.corrcoef(df["baseline_score"], df[col].fillna(0))[0, 1] > 0.99
]

print("1. Direct Leakage Test:")
if not detected_leaks:
    print("   [PASSED] No product flags or label-derived variables leaked into baseline score.")
else:
    print(f"   [FAILED] Detected potential leakage from: {detected_leaks}")

# Check 2: Verify weak pick characteristics programmatically
print("\n2. Weak Pick Audit (Bottom-of-page zero-click & navigational outliers):")
weak_picks = queue[(queue["baseline_rank"].isin([5, 9, 14, 19]))][[
    "baseline_rank", "content_id", "avg_position", "impressions_90d", "ctr", "clicks_90d", "baseline_score"
]]
print(weak_picks.to_string(index=False))

# Check 3: Check baseline precision against declining status as context
base_rate = (df["trend_direction"].str.lower() == "down").mean()
top50_decline_rate = (queue.head(50)["trend_direction"].str.lower() == "down").mean()
print(f"\n3. Baseline Context Alignment:")
print(f"   Dataset Overall Declining Base Rate: {base_rate * 100:.1f}%")
print(f"   Top-50 Baseline Opportunity Declining Rate: {top50_decline_rate * 100:.1f}%")
print(f"   Lift over base rate: {top50_decline_rate / base_rate:.2f}x (honest benchmark for Week 5 model to beat)")
print("=" * 80)

LEAKAGE AND INTEGRITY AUDIT
1. Direct Leakage Test:
   [PASSED] No product flags or label-derived variables leaked into baseline score.

2. Weak Pick Audit (Bottom-of-page zero-click & navigational outliers):
 baseline_rank           content_id  avg_position  impressions_90d  ctr  clicks_90d  baseline_score
             5 content_8451fc6f034d           2.3           272144 0.03          75       1279.0768
             9 content_c8e9d6ab9013           9.7           208678 0.00           0       1043.3900
            14 content_453722754fea           7.6           140079 0.01          16        686.3871
            19 content_4a6607efcb46           2.2           128068 0.01          17        627.5332

3. Baseline Context Alignment:
   Dataset Overall Declining Base Rate: 54.2%
   Top-50 Baseline Opportunity Declining Rate: 50.0%
   Lift over base rate: 0.92x (honest benchmark for Week 5 model to beat)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.